In [ ]:
from aslp_utils import LanceReader, FloatNPYData

In [ ]:
reader = LanceReader('/home/work_nfs16/zhguo/data/dualvc_train_lance/mels', target_cls=FloatNPYData)

In [ ]:
ids = reader.get_ids()
utts = [id.data_id for id in ids]


In [ ]:
utt2spk = {}
bad = []

for utt in utts:
    splits = utt.split('-')
    if len(splits) != 3:
        bad.append(utt)
        continue
    utt2spk[utt] = '-'.join(splits[:2])



In [ ]:
spk2utts = {}
for utt, spk in utt2spk.items():
    if spk not in spk2utts:
        spk2utts[spk] = []
    spk2utts[spk].append(utt)


In [ ]:
spks = list(spk2utts.keys())

In [ ]:
a = reader.get_datas_by_rows([0])[0]

In [ ]:
from tqdm import tqdm

In [ ]:
utt2dur = {}
for row in tqdm(range(len(ids))):
    data = reader.get_datas_by_rows([row])[0]
    dur = data.shape[1] * 80 / 16000
    utt = data.data_id
    utt2dur[utt] = dur



In [ ]:
durs = 0
for utt, dur in utt2dur.items():
    durs += dur
    


从最大的数据中根据filelist长度的百分比筛选数据

In [ ]:
utts = [line.strip() for line in open("../dualvc_train_lance/filelist", 'r').readlines()]

In [ ]:
length = int(len(utts) * 0.5)
import random
utts = random.sample(utts, length)


In [ ]:
open('filelist', 'w').writelines([utt+'\n' for utt in utts])

根据说话人个数来筛选数据

In [ ]:
utts = [line.strip() for line in open("../dualvc_train_lance/filelist", 'r').readlines()]


In [ ]:
utt2spk = {}
bad = []

for utt in utts:
    splits = utt.split('-')
    if len(splits) != 3:
        bad.append(utt)
        continue
    utt2spk[utt] = '-'.join(splits[:2])



In [ ]:
spk2utts = {}
for utt, spk in utt2spk.items():
    if spk not in spk2utts:
        spk2utts[spk] = []
    spk2utts[spk].append(utt)


In [ ]:
spks = list(spk2utts.keys())

In [ ]:
import json
utt2dur = json.load(open("../dualvc_train_lance/utt2dur.json", 'r'))

In [ ]:
def select_utts_with_constraints(utts, utt2spk, spk2utts, utt2dur, N, m):
    # 用于保存最终的filelist
    filelist = []
    total_duration = 0
    selected_spks = set()
    # 获取所有说话人的列表并随机打乱顺序
    all_speakers = list(spk2utts.keys())
    random.shuffle(all_speakers)
    # 遍历说话人直到选出N个说话人且总时长不超过m
    for spk in all_speakers:
        if len(selected_spks) >= N:
            break
        spk_utts = spk2utts[spk]
        spk_total_dur = sum(utt2dur[utt] for utt in spk_utts)
        # 如果该说话人的时长加上去不会超过m，添加这个说话人
        if total_duration + spk_total_dur <= m:
            selected_spks.add(spk)
            filelist.extend(spk_utts)
            total_duration += spk_total_dur
        else:
            # 如果加上该说话人超过了m，则从这个说话人的utterances中选一部分
            spk_utts_sorted = sorted(spk_utts, key=lambda utt: utt2dur[utt])
            for utt in spk_utts_sorted:
                utt_dur = utt2dur[utt]
                if total_duration + utt_dur <= m:
                    filelist.append(utt)
                    total_duration += utt_dur
                else:
                    break
    return filelist, total_duration, len(selected_spks)

In [ ]:
N = 450
m = 125 * 60 * 60

In [ ]:
import random
filelist, total_duration, len_selected_spks = select_utts_with_constraints(utts, utt2spk, spk2utts, utt2dur, N, m)

In [ ]:
open('filelist', 'w').writelines([utt+'\n' for utt in filelist])

根据audioreader获得utt2dur

In [ ]:
cn_reader = LanceReader('/home/work_nfs16/zhguo/data/hq_cn_lance_data', target_cls=AudioData)
en_reader = LanceReader('/home/work_nfs16/zhguo/data/hq_en', target_cls=AudioData)
rest_reader = LanceReader('/home/work_nfs16/zhguo/data/hq_cn_mix_rest', target_cls=AudioData)

In [ ]:
utt2dur = {}

In [ ]:
for i in tqdm(range(len(cn_reader.get_ids()))):
    data = cn_reader.get_datas_by_rows([i], ['data_id', 'duration'])[0]
    utt2dur[data.data_id] = data.duration

In [ ]:
sum(utt2dur.values())


In [ ]:
for i in tqdm(range(len(rest_reader.get_ids()))):
    data = rest_reader.get_datas_by_rows([i], ['data_id', 'duration'])[0]
    utt2dur[data.data_id] = data.duration

In [ ]:
for i in tqdm(range(len(en_reader.get_ids()))):
    data = en_reader.get_datas_by_rows([i], ['data_id', 'duration'])[0]
    utt2dur[data.data_id] = data.duration



In [ ]:
utt2dur_fix = {}
for utt, dur in utt2dur.items():
    utt2dur_fix[utt] = int(dur)



In [ ]:
import json
json.dump(utt2dur_fix, open('/home/work_nfs16/zhguo/data/dualvc_data/dualvc_train_lance_hq_all/utt2dur.json', 'w'), indent=4)
